<a href="https://colab.research.google.com/github/matheusgr76/matheusgr76/blob/main/NBA_data_extractor_1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
import cloudscraper
import time
from io import StringIO
from bs4 import BeautifulSoup
from nba_api.stats.endpoints import leaguestandings, leaguedashplayerstats, leaguedashteamstats

# =================================================================
# CONFIGURAÇÕES E MAPEAMENTO MESTRE (30 TIMES)
# =================================================================
API_KEY = 'fb80551fe2e719b0606d59118e2bd0d5'

# Mapeamento completo para garantir que NBA API e ESPN falem a mesma língua
TEAM_MASTER_MAP = {
    'Atlanta Hawks': {'abbr': 'ATL', 'slug': 'atl/atlanta-hawks'},
    'Boston Celtics': {'abbr': 'BOS', 'slug': 'bos/boston-celtics'},
    'Brooklyn Nets': {'abbr': 'BKN', 'slug': 'bkn/brooklyn-nets'},
    'Charlotte Hornets': {'abbr': 'CHA', 'slug': 'cha/charlotte-hornets'},
    'Chicago Bulls': {'abbr': 'CHI', 'slug': 'chi/chicago-bulls'},
    'Cleveland Cavaliers': {'abbr': 'CLE', 'slug': 'cle/cleveland-cavaliers'},
    'Dallas Mavericks': {'abbr': 'DAL', 'slug': 'dal/dallas-mavericks'},
    'Denver Nuggets': {'abbr': 'DEN', 'slug': 'den/denver-nuggets'},
    'Detroit Pistons': {'abbr': 'DET', 'slug': 'det/detroit-pistons'},
    'Golden State Warriors': {'abbr': 'GSW', 'slug': 'gs/golden-state-warriors'},
    'Houston Rockets': {'abbr': 'HOU', 'slug': 'hou/houston-rockets'},
    'Indiana Pacers': {'abbr': 'IND', 'slug': 'ind/indiana-pacers'},
    'LA Clippers': {'abbr': 'LAC', 'slug': 'lac/la-clippers'},
    'Los Angeles Clippers': {'abbr': 'LAC', 'slug': 'lac/la-clippers'},
    'Los Angeles Lakers': {'abbr': 'LAL', 'slug': 'lal/los-angeles-lakers'},
    'Memphis Grizzlies': {'abbr': 'MEM', 'slug': 'mem/memphis-grizzlies'},
    'Miami Heat': {'abbr': 'MIA', 'slug': 'mia/miami-heat'},
    'Milwaukee Bucks': {'abbr': 'MIL', 'slug': 'mil/milwaukee-bucks'},
    'Minnesota Timberwolves': {'abbr': 'MIN', 'slug': 'min/minnesota-timberwolves'},
    'New Orleans Pelicans': {'abbr': 'NOP', 'slug': 'no/new-orleans-pelicans'},
    'New York Knicks': {'abbr': 'NYK', 'slug': 'ny/new-york-knicks'},
    'Oklahoma City Thunder': {'abbr': 'OKC', 'slug': 'okc/oklahoma-city-thunder'},
    'Orlando Magic': {'abbr': 'ORL', 'slug': 'orl/orlando-magic'},
    'Philadelphia 76ers': {'abbr': 'PHI', 'slug': 'phi/philadelphia-76ers'},
    'Phoenix Suns': {'abbr': 'PHX', 'slug': 'phx/phoenix-suns'},
    'Portland Trail Blazers': {'abbr': 'POR', 'slug': 'por/portland-trail-blazers'},
    'Sacramento Kings': {'abbr': 'SAC', 'slug': 'sac/sacramento-kings'},
    'San Antonio Spurs': {'abbr': 'SAS', 'slug': 'sas/san-antonio-spurs'},
    'Toronto Raptors': {'abbr': 'TOR', 'slug': 'tor/toronto-raptors'},
    'Utah Jazz': {'abbr': 'UTA', 'slug': 'uta/utah-jazz'},
    'Washington Wizards': {'abbr': 'WAS', 'slug': 'was/washington-wizards'}
}

scraper = cloudscraper.create_scraper()
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

# =================================================================
# FUNÇÕES DE APOIO
# =================================================================

def get_espn_schedule(team_abbr, season=2026):
    # Busca o slug correto no mapeamento mestre
    slug = "n/a"
    for name, data in TEAM_MASTER_MAP.items():
        if data['abbr'] == team_abbr:
            slug = data['slug']
            break

    if slug == "n/a": slug = team_abbr.lower() # Fallback

    url = f"https://www.espn.com/nba/team/schedule/_/name/{slug}/season/{season}"
    try:
        res = scraper.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')
        table = soup.find('table')
        if not table: return pd.DataFrame()
        df = pd.read_html(StringIO(str(table)))[0]
        df.columns = [str(i) for i in range(len(df.columns))]
        return df[df['0'].str.contains(',', na=False)].copy()
    except: return pd.DataFrame()

def process_espn_logic(team_abbr, opp_abbr):
    df = get_espn_schedule(team_abbr, 2026)
    if df.empty: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    forma = df[df['2'].str.contains('W|L', na=False)].tail(5)
    agenda = df[~df['2'].str.contains('W|L', na=False)].head(5)

    # H2H: Filtra por oponente e garante que já houve resultado
    h2h = df[(df['1'].str.contains(opp_abbr, case=False, na=False)) & (df['2'].str.contains('W|L', na=False))]

    return forma, h2h, agenda

def get_injury_report_espn():
    try:
        url = "https://www.espn.com/nba/injuries"
        res = scraper.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(res.text, 'html.parser')
        injuries = []
        for table in soup.find_all('div', class_='Table__Title'):
            team = table.text.strip()
            rows = table.find_next('table').find_all('tr')[1:]
            for r in rows:
                cols = r.find_all('td')
                if len(cols) >= 3:
                    injuries.append({'Team': team, 'Player': cols[0].text.strip(), 'Status': cols[2].text.strip(), 'Injury': cols[3].text.strip() if len(cols)>3 else "N/A"})
        return pd.DataFrame(injuries)
    except: return pd.DataFrame()

def get_referee_stats(names_list):
    try:
        url = "https://www.oddsshark.com/nba/referee-handicapping-statistics"
        df = pd.read_html(StringIO(scraper.get(url).text))[0]
        df.columns = ['Referee', 'Home_Scoring', 'Away_Scoring', 'Home_Wins', 'Home_Losses', 'ATS_H_W', 'ATS_H_L', 'ATS_H_P', 'H_Fouls', 'A_Fouls'][:len(df.columns)]
        clean_names = [n.split('(')[0].strip() for n in names_list]
        return df[df['Referee'].str.contains('|'.join(clean_names), case=False, na=False)]
    except: return pd.DataFrame()

# =================================================================
# ENGINE FINAL
# =================================================================

def engine_nba_v13_final_revised():
    ref_assignments = {}
    try:
        r_ref = requests.get("https://official.nba.com/referee-assignments/", headers=headers)
        soup_ref = BeautifulSoup(r_ref.text, 'html.parser')
        rows = soup_ref.find('table').find_all('tr')[1:]
        for row in rows:
            cols = [td.text.strip() for td in row.find_all('td')]
            if len(cols) >= 4:
                ref_assignments[cols[0]] = [cols[1], cols[2], cols[3]]
    except: pass

    inj_df = get_injury_report_espn()
    try:
        std_raw = leaguestandings.LeagueStandings(season='2025-26').get_data_frames()[0]
        std_raw['Team'] = std_raw['TeamCity'] + " " + std_raw['TeamName']
        std_clean = std_raw[['Team', 'PlayoffRank', 'Record', 'WinPCT']]
        ts_raw = leaguedashteamstats.LeagueDashTeamStats(season='2025-26', measure_type_detailed_defense='Advanced').get_data_frames()[0]
        ts_clean = ts_raw[['TEAM_NAME', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'PACE']]
    except: std_clean, ts_clean = pd.DataFrame(), pd.DataFrame()

    try:
        res = requests.get(f'https://api.the-odds-api.com/v4/sports/basketball_nba/odds/?apiKey={API_KEY}&regions=us,eu&markets=h2h,spreads,totals').json()
        all_games = {g['id']: g for g in res}
    except: all_games = {}

    for gid, game in all_games.items():
        home, away = game['home_team'], game['away_team']

        # Obtém abreviações usando o mapa mestre
        h_abbr = TEAM_MASTER_MAP.get(home, {'abbr': home[:3].upper()})['abbr']
        a_abbr = TEAM_MASTER_MAP.get(away, {'abbr': away[:3].upper()})['abbr']

        print(f"\n{'='*130}\n MATCHUP: {away.upper()} @ {home.upper()} \n{'='*130}")

        # [1] ODDS
        print("\n[1. ODDS COMPARISON]")
        for bm in game['bookmakers']:
            if bm['key'] in ['fanduel', 'draftkings', 'pinnacle']:
                m = {market['key']: market['outcomes'] for market in bm['markets']}
                ml = next((o['price'] for o in m.get('h2h', []) if o['name'] == home), "N/A")
                sp = next((o['point'] for o in m.get('spreads', []) if o['name'] == home), "N/A")
                tl = next((o['point'] for o in m.get('totals', []) if o['name'] == 'Over'), "N/A")
                print(f" {bm['title']:14} | ML_H: {ml:<5} | Spread: {sp:<5} | Total: {tl}")

        # [2] STANDINGS & STATS
        print("\n[2. STANDINGS & TEAM STATS]")
        if not std_clean.empty:
            print(std_clean[std_clean['Team'].str.contains(f"{home.split()[-1]}|{away.split()[-1]}", case=False)].to_string(index=False))
        if not ts_clean.empty:
            print("-" * 60)
            print(ts_clean[ts_clean['TEAM_NAME'].str.contains(f"{home.split()[-1]}|{away.split()[-1]}", case=False)].to_string(index=False))

        # [3] INJURIES
        print("\n[3. INJURY REPORT]")
        if not inj_df.empty:
            m_inj = inj_df[inj_df['Team'].str.contains(f"{home.split()[-1]}|{away.split()[-1]}", case=False)]
            print(m_inj[['Player', 'Status', 'Injury']].to_string(index=False) if not m_inj.empty else "Sem lesões.")

        # [4] HISTORY, FORM & SCHEDULE
        print("\n[4. HISTORY, FORM & SCHEDULE (ESPN)]")
        f_a, h2h_a, a_a = process_espn_logic(a_abbr, h_abbr)
        f_h, h2h_h, a_h = process_espn_logic(h_abbr, a_abbr)

        print(f"\nFORM (Últimos 5) - {a_abbr}:\n{f_a[['0', '1', '2']].rename(columns={'0':'DATA','1':'OPP','2':'RES'}).to_string(index=False) if not f_a.empty else 'N/A'}")
        print(f"\nFORM (Últimos 5) - {h_abbr}:\n{f_h[['0', '1', '2']].rename(columns={'0':'DATA','1':'OPP','2':'RES'}).to_string(index=False) if not f_h.empty else 'N/A'}")

        print("\nHEAD-TO-HEAD (Temporada Atual):")
        h2h_all = pd.concat([h2h_a, h2h_h]).drop_duplicates(subset=['0', '2'])
        print(h2h_all[['0', '1', '2']].rename(columns={'0':'DATA','1':'MATCHUP','2':'RES'}).to_string(index=False) if not h2h_all.empty else "N/A")

        print(f"\nSCHEDULE (Próximos 5) - {a_abbr}:\n{a_a[['0', '1', '2']].rename(columns={'0':'DATA','1':'OPP','2':'HORA'}).to_string(index=False) if not a_a.empty else 'N/A'}")
        print(f"\nSCHEDULE (Próximos 5) - {h_abbr}:\n{a_h[['0', '1', '2']].rename(columns={'0':'DATA','1':'OPP','2':'HORA'}).to_string(index=False) if not a_h.empty else 'N/A'}")

        # [5] REFEREE STATS
        print("\n[5. REFEREE ANALYSIS]")
        trio = next((v for k, v in ref_assignments.items() if (h_abbr in k or home.split()[-1] in k)), None)
        if trio:
            print(f"Trio Escalado: {trio}")
            print(get_referee_stats(trio).to_string(index=False))
        else: print("Escala individual não disponível.")

        # [6] PLAYER STATS
        print("\n[6. TOP 10 PLAYER STATS]")
        try:
            base = leaguedashplayerstats.LeagueDashPlayerStats(season='2025-26').get_data_frames()[0]
            adv = leaguedashplayerstats.LeagueDashPlayerStats(season='2025-26', measure_type_detailed_defense='Advanced').get_data_frames()[0]
            merged = pd.merge(base[['PLAYER_NAME', 'TEAM_ABBREVIATION', 'MIN', 'PTS', 'FGA']], adv[['PLAYER_NAME', 'USG_PCT']], on='PLAYER_NAME')
            p_stats = merged[merged['TEAM_ABBREVIATION'].isin([h_abbr, a_abbr])].sort_values(by='MIN', ascending=False).groupby('TEAM_ABBREVIATION').head(10)
            print(p_stats.rename(columns={'PLAYER_NAME': 'Player', 'TEAM_ABBREVIATION': 'Team', 'USG_PCT': 'Usage'}).to_string(index=False))
        except: pass
        time.sleep(1)

    # --- LISTA GLOBAL DE ÁRBITROS ---
    print("\n" + "="*130)
    print(" GLOBAL REFEREE ASSIGNMENTS (OFFICIAL.NBA.COM) ".center(130, '='))
    print("="*130)
    if ref_assignments:
        for jogo, trio in ref_assignments.items():
            print(f"Jogo: {jogo:25} | Trio: {', '.join(trio)}")

    # --- LINEUPS GLOBAIS ---
    print("\n" + "#"*130)
    print(" GLOBAL NBA LINEUPS (BASKETBALL MONSTER) ".center(130, '#'))
    print("#"*130)
    try:
        l_page = scraper.get("https://basketballmonster.com/nbalineups.aspx").text
        l_soup = BeautifulSoup(l_page, 'html.parser')
        for row in l_soup.find_all('tr'):
            cols = row.find_all('td')
            if 'class' in row.attrs and 'lineup-header' in row['class']:
                teams = row.find_all('a')
                if len(teams) >= 2:
                    print(f"\n● {teams[0].text.strip()} @ {teams[1].text.strip()} ".ljust(100, '-'))
                    print(f"{'POS':<5} | {'AWAY TEAM':<25} | {'HOME TEAM':<25}")
            if len(cols) >= 3 and cols[0].text.strip() in ['PG', 'SG', 'SF', 'PF', 'C']:
                print(f"{cols[0].text.strip():<5} | {cols[1].get_text(strip=True).split('News')[0]:<25} | {cols[2].get_text(strip=True).split('News')[0]:<25}")
    except: pass

engine_nba_v13_final_revised()


 MATCHUP: UTAH JAZZ @ CLEVELAND CAVALIERS 

[1. ODDS COMPARISON]
 FanDuel        | ML_H: 1.18  | Spread: -13.0 | Total: 250.5
 DraftKings     | ML_H: 1.18  | Spread: -12.5 | Total: 249.5
 Pinnacle       | ML_H: 1.2   | Spread: -13.0 | Total: 249.5

[2. STANDINGS & TEAM STATS]
               Team  PlayoffRank Record  WinPCT
Cleveland Cavaliers            7  22-18   0.550
          Utah Jazz           13  13-25   0.342
------------------------------------------------------------
          TEAM_NAME  OFF_RATING  DEF_RATING  NET_RATING   PACE
Cleveland Cavaliers       116.8       114.3         2.5 102.23
          Utah Jazz       114.0       122.0        -8.0 102.68

[3. INJURY REPORT]
          Player Status     Injury
       Dean Wade Jan 14        Out
       Max Strus Feb 11        Out
Chris Livingston Jan 18        Out
   Georges Niang Jan 19        Out
 Elijah Harkless Jan 13 Day-To-Day
  Walker Kessler  Oct 1        Out

[4. HISTORY, FORM & SCHEDULE (ESPN)]

FORM (Últimos 5) - UTA:
